# CVS File

In [ ]:
import io
import pandas as pd
from google.colab import files

# Ask the user for their details
student_name = input("Enter your name: ")
course_name = input("Enter your Course & Major (e.g., BSCS): ")
year_level = input("Enter your Year Level (e.g., 1st Year): ")

# Display the personalized header and grade report
print("\n" + "="*40)
print(f"GRADE REPORT FOR: {student_name.strip().upper()}")
print(f"COURSE & MAJOR:   {course_name.strip().upper()}")
print(f"YEAR LEVEL:       {year_level.strip().upper()}")
print("="*40 + "\n")

# 1. Trigger the file upload prompt
uploaded = files.upload()

# Helper function to find the header row index and read df correctly
def read_grades_df(file_bytes, is_excel=False):
    # Read raw data first to inspect structure
    if is_excel:
        raw_df = pd.read_excel(io.BytesIO(file_bytes), header=None)
    else:
        raw_df = pd.read_csv(io.BytesIO(file_bytes), header=None)

    # Find row index containing typical headers
    header_row_idx = 0
    for i in range(len(raw_df)):
        row_vals = [str(val).strip().lower() for val in raw_df.iloc[i].values if pd.notna(val)]
        if 'units' in row_vals or 'subject' in row_vals or 'grade' in row_vals:
            header_row_idx = i
            break

    # Re-read utilizing the correct header row offset
    if is_excel:
        return pd.read_excel(io.BytesIO(file_bytes), skiprows=header_row_idx)
    else:
        return pd.read_csv(io.BytesIO(file_bytes), skiprows=header_row_idx)

# 2. Dynamically get the name of the uploaded file
if uploaded:
    file_name = list(uploaded.keys())[0]

    # Check file extension to read it correctly
    is_excel = file_name.endswith('.xlsx') or file_name.endswith('.xls')
    df = read_grades_df(uploaded[file_name], is_excel=is_excel)

    # Normalize column names by stripping whitespaces
    df.columns = [str(col).strip() for col in df.columns]

    # 3. View the full data
    display(df)
else:
    print('No file was uploaded.')

# Calculate and print grade report statistics
if 'df' in globals() and df is not None:
    # Ensure we target the right columns case-insensitively if needed
    units_col = next((col for col in df.columns if col.lower() == 'units'), 'Units')
    grade_col = next((col for col in df.columns if col.lower() == 'grade'), 'Grade')

    # 1. Total Number of Subjects
    num_subjects = len(df)

    # 2. Total Units (ensure they are numeric)
    df[units_col] = pd.to_numeric(df[units_col], errors='coerce')
    total_units = df[units_col].sum()

    # 3. GWA (Grade Weighted Average)
    # Ensure Grade is numeric
    df[grade_col] = pd.to_numeric(df[grade_col], errors='coerce')

    # Calculate GWA: sum(Grade * Units) / sum(Units)
    weighted_sum = (df[grade_col] * df[units_col]).sum()
    gwa = weighted_sum / total_units if total_units > 0 else 0

    print("="*40)
    print("GRADE SUMMARY")
    print("="*40)
    print(f"Student Name:                   {student_name.strip().upper()}")
    print(f"Course & Major:                 {course_name.strip().upper()}")
    print(f"Year Level:                     {year_level.strip().upper()}")
    print(f"Number of Subjects:             {num_subjects}")
    print(f"Total Units:                    {total_units:.1f}")
    print(f"GWA Grade Weighted Average:     {gwa:.2f}")
    print("="*40)
else:
    print("DataFrame 'df' is not loaded. Please upload the file in the previous step first.")